# 다봐요 (Dabwayo) — LaMa 워터마크 제거 (Colab)

AI 영상(Veo/Gemini ✦, ModelScope의 shutterstock 등)에 박힌 **정적 워터마크**를
**LaMa(딥러닝 인페인팅)** 로 자연스럽게 제거합니다. 고전 OpenCV inpaint와 달리
복잡·움직이는 배경에서도 그럴듯한 텍스처를 *생성*해 채웁니다.

**사용법:** Runtime → (선택) GPU(T4) → **Run all**. 영상 업로드 → 워터마크
네모(REGION) 확인 → 실행하면 오디오까지 보존된 결과가 자동 다운로드됩니다.
GPU면 빠르고, CPU여도 작은 영역이라 동작합니다.

## 1. 설치 (torch는 Colab에 기본 내장)

In [ ]:
%pip -q install simple-lama-inpainting imageio imageio-ffmpeg opencv-python-headless
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 2. 영상 업로드

In [ ]:
from google.colab import files
import imageio.v2 as imageio
up = files.upload()                       # Veo/생성 mp4 선택
SRC = list(up.keys())[0]
_r = imageio.get_reader(SRC); _m = _r.get_meta_data()
first = _r.get_data(0); H, W = first.shape[:2]
FPS = float(_m.get('fps', 24))
print(f'{SRC}: {W}x{H} @ {FPS}fps')

## 3. 워터마크 영역 지정 + 미리보기
`REGION = [x, y, w, h]` (픽셀). 빨간 네모가 워터마크를 덮도록 숫자를 조정하세요.
Gemini ✦(1280×720)의 기본값은 우하단입니다.

In [ ]:
REGION = [1132, 568, 60, 62]              # [x, y, w, h] — 본인 영상에 맞게 조정
PAD, FEATHER, DILATE = 48, 4, 9           # ROI 여백 / 경계 블렌딩 / 마스크 확장

import numpy as np, cv2
from PIL import Image
from IPython.display import display
x,y,w,h = REGION
prev = first.copy()
cv2.rectangle(prev, (x,y), (x+w,y+h), (255,0,0), 3)
crop = prev[max(0,y-90):y+h+90, max(0,x-120):x+w+120]
display(Image.fromarray(crop))            # 빨간 네모가 워터마크를 덮는지 확인

## 4. LaMa로 프레임별 제거

In [ ]:
from simple_lama_inpainting import SimpleLama
from PIL import Image, ImageFilter
import imageio_ffmpeg, tempfile, numpy as np, cv2

lama = SimpleLama()                        # 첫 실행 시 모델 가중치 다운로드

x,y,w,h = REGION
x0,y0 = max(0,x-PAD), max(0,y-PAD)
x1,y1 = min(W,x+w+PAD), min(H,y+h+PAD)
mask = np.zeros((H,W), np.uint8); mask[y:y+h, x:x+w] = 255
if DILATE>0: mask = cv2.dilate(mask, np.ones((DILATE,DILATE),np.uint8))
roi_mask = mask[y0:y1, x0:x1]
roi_mask_pil = Image.fromarray(roi_mask, 'L')
blend = roi_mask.astype(np.float32)/255.0
if FEATHER>0:
    blend = cv2.GaussianBlur(blend, (0, 0), float(FEATHER))
blend = blend[...,None]

tmp = tempfile.mktemp(suffix='.mp4')
wr = imageio.get_writer(tmp, fps=FPS, codec='libx264', quality=9, macro_block_size=None)
rdr = imageio.get_reader(SRC)
for i, frame in enumerate(rdr):
    roi = frame[y0:y1, x0:x1]
    res = lama(Image.fromarray(roi,'RGB'), roi_mask_pil)
    res = np.asarray(res.convert('RGB').resize((roi.shape[1], roi.shape[0])), np.float32)/255.0
    out = frame.astype(np.float32)/255.0
    out[y0:y1, x0:x1, :3] = res*blend + out[y0:y1, x0:x1, :3]*(1-blend)
    wr.append_data((np.clip(out,0,1)*255).astype(np.uint8))
    if i % 24 == 0: print('frame', i)
wr.close(); rdr.close()
print('inpainting done ->', tmp)

## 5. 원본 오디오 보존(mux) + 다운로드

In [ ]:
import subprocess, imageio_ffmpeg
ff = imageio_ffmpeg.get_ffmpeg_exe()
OUT = 'dewatermarked_lama.mp4'
has_audio = subprocess.run([ff,'-i',SRC], capture_output=True, text=True).stderr.find('Audio:') != -1
if has_audio:
    subprocess.run([ff,'-y','-v','error','-i',tmp,'-i',SRC,'-map','0:v','-map','1:a',
                    '-c:v','copy','-c:a','aac','-b:a','128k','-shortest',OUT], check=True)
else:
    subprocess.run([ff,'-y','-v','error','-i',tmp,'-c','copy',OUT], check=True)
print('wrote', OUT)
from google.colab import files; files.download(OUT)